<a href="https://colab.research.google.com/github/ANEESAAYAZ/internship_feature_engineering/blob/main/internship_FeatureEngineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 60)
SNAPSHOT=pd.Timestamp("2024-05-01")

In [5]:
df=pd.read_csv("properties_clean.csv",parse_dates=["listed_date"])
print(f"loaded dataset has {df.shape[0]} rows and {df.shape[1]} columns")

loaded dataset has 3000 rows and 42 columns


In [6]:
df["sold_fast"]=np.where(df["listing_status"]=="Active",np.nan,(df["days_on_market"]<=30).astype(float))


In [7]:
df["sold_fast"][:10]

,sold_fast
0,0.0
1,0.0
2,0.0
3,1.0
4,1.0
5,1.0
6,0.0
7,0.0
8,0.0
9,0.0


In [9]:
df["sold_fast"][:10].tolist()

[0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0]

In [14]:
print(f"\n Target 1 - price_aed : Regression {df['price_aed'].notna().sum()} rows usable")
print(f"\n Target 2 - sold_fast : Classification {df['sold_fast'].notna().sum()} rpws usable"
      f" ({df["sold_fast"].mean():.1%}) positive")


 Target 1 - price_aed : Regression 3000 rows usable

 Target 2 - sold_fast : Classification 2563 rpws usable (38.7%) positive


**Combine**

In [15]:
df["price_per_sqft"]=df["price_aed"]/df["built_up_area_sqft"]
df["rental_yield_pct"]=100*df["annual_rent_aed"]/df["price_aed"]
df["bath_per_bed"]=df["bathrooms"]/df["bedrooms"].replace(0,1)
df["service_charge_annual"]=df["service_charge_aed_sqft"]*df["built_up_area_sqft"]
df["sqft_per_bed"]=df["built_up_area_sqft"]/df["bedrooms"].replace(0,1)
df["plot_ratio"]=np.where(df["has_plot_area_sqft"]==1,df["plot_area_sqft"]/df["built_up_area_sqft"],np.nan)

In [17]:
print("\n--------price per sqft by emirate (the number every agent quote)--------")
print(df.groupby("emirate")["price_per_sqft"].median().round(0).sort_values())

print("\n----------rental yield plausible-----------")
print(df["rental_yield_pct"].describe().round(2))

print("\nUAE market reality:5%-9% is normal,above 11% is bad data")


--------price per sqft by emirate (the number every agent quote)--------
emirate
Ajman              390.0
Sharjah            923.0
Abu Dhabi         1375.0
Ras Al Khaimah    1426.0
Dubai             1740.0
Name: price_per_sqft, dtype: float64

----------rental yield plausible-----------
count    1554.00
mean        7.06
std         1.99
min         0.45
25%         5.79
50%         6.74
75%         8.02
max        19.61
Name: rental_yield_pct, dtype: float64

UAE market reality:5%-9% is normal,above 11% is bad data


**Decode**

In [18]:
df["listing_month"]=df["listed_date"].dt.month
df["listing_quarter"]=df["listed_date"].dt.quarter
df["listing_year"]=df["listed_date"].dt.year
df["listing_dayofweek"]=df["listed_date"].dt.dayofweek
df["days_since_listed"]=(SNAPSHOT - df["listed_date"]).dt.days
df["is_summer_listing"]=df["listing_month"].isin([6,7,8]).astype(int)
df["is_winter_listing"]=df["listing_month"].isin([12,1,2]).astype(float)
df["month_sin"]=np.sin(2*np.pi*df["listing_month"]/12)
df["month_cos"]=np.cos(2*np.pi*df["listing_month"]/12)

In [19]:

print("\n------listing per month collection------")
print(df["listed_date"].dt.to_period("M").value_counts().sort_index().tail(10).to_string())


------listing per month collection------
listed_date
2023-07      19
2023-08      32
2023-09      41
2023-10      83
2023-11     139
2023-12     142
2024-01     261
2024-02     478
2024-03     727
2024-04    1040
Freq: M


In [ ]:
settled=df[df["sold_fast"].notna()]
